# Running Multiple Agent Calls in Parallel

This notebook demonstrates how to use `asyncio` with the Anthropic async client to run multiple API calls concurrently. Instead of processing items one at a time (sequential), we fire off all requests at once (concurrent) and collect the results when they're all done.

**WHY THIS MATTERS:**
- If you have 10 property listings to analyze and each takes 3 seconds:
  - Sequential: 10 x 3s = 30 seconds total
  - Concurrent: ~3 seconds total (all running at the same time)

**WHEN TO USE:**
- Processing a batch of independent items (each doesn't depend on another)
- Running the same analysis with different inputs
- Any time you'd otherwise loop through items one at a time

**WHEN NOT TO USE:**
- When results depend on each other (step B needs step A's output)
- When you need to respect strict API rate limits
- When order of execution matters

## Setup

Install the Anthropic SDK and initialize the **async** client. Note: We use `AsyncAnthropic` instead of `Anthropic` for async support.

In [ ]:
!pip install anthropic

In [ ]:
import os
import json
import asyncio
import anthropic

# Setup: Initialize the ASYNC Anthropic client
# Note: We use AsyncAnthropic instead of Anthropic for async support.
client = anthropic.AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Sample Data: Properties to Analyze

Imagine these came from a CSV export, MLS feed, or database query. Each property is independent -- the analysis of one doesn't affect another. This makes them perfect for parallel processing.

In [ ]:
PROPERTIES = [
    {
        "address": "123 Kailua Rd, Kailua, HI",
        "beds": 3,
        "baths": 2,
        "sqft": 1400,
        "asking_price": 1250000,
        "features": "Ocean view, renovated kitchen, single-story",
    },
    {
        "address": "456 Manoa Rd, Honolulu, HI",
        "beds": 4,
        "baths": 3,
        "sqft": 1800,
        "asking_price": 1100000,
        "features": "Large yard, mountain views, needs cosmetic updates",
    },
    {
        "address": "789 Kalanianaole Hwy, Hawaii Kai, HI",
        "beds": 3,
        "baths": 2.5,
        "sqft": 1550,
        "asking_price": 1380000,
        "features": "Marina access, covered lanai, two-car garage",
    },
    {
        "address": "321 Ala Moana Blvd #1205, Honolulu, HI",
        "beds": 2,
        "baths": 2,
        "sqft": 1050,
        "asking_price": 850000,
        "features": "Condo, ocean view, pool, doorman, walkable to shopping",
    },
    {
        "address": "555 North Shore Dr, Haleiwa, HI",
        "beds": 3,
        "baths": 2,
        "sqft": 1200,
        "asking_price": 975000,
        "features": "Steps to beach, cottage style, short-term rental potential",
    },
]

## System Prompt for Property Analysis

In [ ]:
SYSTEM_PROMPT = """You are a real estate investment analyst for Hawaii properties.

Given a property listing, provide a brief analysis covering:
1. Price per square foot assessment
2. Key value drivers from the listed features
3. Potential concerns or risks
4. Quick verdict: Fairly priced / Overpriced / Underpriced

Keep your analysis concise -- 4-6 sentences max. Be direct and data-driven."""

## Single Property Analysis (Async)

This function analyzes ONE property. It's an async function, meaning it can be paused while waiting for the API response, allowing other calls to run.

In [ ]:
async def analyze_property(property_data: dict) -> dict:
    """
    Analyze a single property using the Anthropic API.
    Returns a dict with the property address and the agent's analysis.
    """
    # Build an evidence-based prompt with the specific property data
    user_message = f"""Analyze this property listing:

Address: {property_data['address']}
Bedrooms: {property_data['beds']}
Bathrooms: {property_data['baths']}
Square Feet: {property_data['sqft']}
Asking Price: ${property_data['asking_price']:,}
Price/SqFt: ${property_data['asking_price'] / property_data['sqft']:,.0f}
Features: {property_data['features']}

Provide your analysis."""

    try:
        # Make the async API call
        response = await client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=500,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )

        # Extract the text response
        analysis = response.content[0].text

        return {
            "address": property_data["address"],
            "asking_price": property_data["asking_price"],
            "analysis": analysis,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "status": "success",
        }

    except Exception as e:
        # If one call fails, we still want the others to complete.
        # Return the error instead of crashing the whole batch.
        return {
            "address": property_data["address"],
            "asking_price": property_data["asking_price"],
            "analysis": None,
            "error": str(e),
            "status": "error",
        }

## Batch Analysis (Concurrent)

This is where the magic happens. `asyncio.gather()` takes multiple coroutines and runs them ALL at the same time. Instead of waiting for each one to finish before starting the next, they all run concurrently.

In [ ]:
async def analyze_all_properties(properties: list[dict]) -> list[dict]:
    """
    Analyze all properties concurrently.
    Returns a list of results in the same order as the input.
    """
    print(f"Starting concurrent analysis of {len(properties)} properties...\n")

    # Create a list of coroutines (one per property)
    tasks = [analyze_property(prop) for prop in properties]

    # Run all tasks concurrently and wait for ALL to complete
    # asyncio.gather() returns results in the same order as the input tasks
    results = await asyncio.gather(*tasks)

    return results

## Rate-Limited Batch Analysis (Concurrent with Throttling)

In production, you may need to respect API rate limits. This version uses a semaphore to limit how many calls run at the same time.

In [ ]:
async def analyze_with_rate_limit(properties: list[dict], max_concurrent: int = 3) -> list[dict]:
    """
    Analyze properties concurrently, but limit how many run at once.
    Useful when you need to respect API rate limits.

    Args:
        properties: List of property dicts to analyze
        max_concurrent: Maximum number of API calls to run simultaneously
    """
    # A semaphore limits how many coroutines can run the protected section
    # at the same time. If max_concurrent=3, only 3 API calls happen at once.
    semaphore = asyncio.Semaphore(max_concurrent)

    async def limited_analyze(prop: dict) -> dict:
        async with semaphore:
            # Only max_concurrent calls will be inside this block at a time
            return await analyze_property(prop)

    print(f"Starting rate-limited analysis ({max_concurrent} concurrent) of {len(properties)} properties...\n")

    tasks = [limited_analyze(prop) for prop in properties]
    results = await asyncio.gather(*tasks)

    return results

## Display Results

Pretty-print the analysis results and show token usage summary for cost tracking.

In [ ]:
def display_results(results: list[dict]) -> None:
    """Pretty-print the analysis results."""
    print(f"\n{'='*70}")
    print("PROPERTY ANALYSIS RESULTS")
    print(f"{'='*70}\n")

    total_input_tokens = 0
    total_output_tokens = 0

    for i, result in enumerate(results, 1):
        print(f"--- Property {i}: {result['address']} ---")
        print(f"Asking Price: ${result['asking_price']:,}")

        if result["status"] == "success":
            print(f"Analysis:\n{result['analysis']}")
            total_input_tokens += result["input_tokens"]
            total_output_tokens += result["output_tokens"]
        else:
            print(f"ERROR: {result.get('error', 'Unknown error')}")

        print()

    # Token usage summary -- important for cost tracking
    print(f"{'='*70}")
    print("TOKEN USAGE SUMMARY")
    print(f"Total input tokens:  {total_input_tokens:,}")
    print(f"Total output tokens: {total_output_tokens:,}")
    print(f"Total tokens:        {total_input_tokens + total_output_tokens:,}")
    print(f"{'='*70}")

## Run the Full Batch Analysis

We use `await` to start the async event loop. In a Jupyter notebook, the event loop is already running, so we use `await` directly instead of `asyncio.run()`.

In [ ]:
import time

# Option 1: Full concurrent (all at once)
start = time.time()
results = await analyze_all_properties(PROPERTIES)
elapsed = time.time() - start
print(f"\nFull concurrent: {elapsed:.1f}s for {len(PROPERTIES)} properties")

display_results(results)

## Rate-Limited Mode (Optional)

Uncomment and run the cell below to try the rate-limited version, which limits concurrent API calls to 3 at a time.

In [ ]:
# Option 2: Rate-limited concurrent (max 3 at a time)
# Uncomment the lines below to try rate-limited mode:
#
# start = time.time()
# results = await analyze_with_rate_limit(PROPERTIES, max_concurrent=3)
# elapsed = time.time() - start
# print(f"\nRate-limited (3 concurrent): {elapsed:.1f}s for {len(PROPERTIES)} properties")
# display_results(results)